# Câu 3a. Xây dựng vector đặc trưng văn bản: Bag-of-Words (BOW)
**Input:** `../Data_Preprocessing` (kết quả Câu 1), mỗi file là 1 văn bản
**Output:** ma trận BOW (số văn bản × số từ trong từ điển), lưu file `BOW.csv`
*(Tham khảo Ví dụ 4.2a, 4.2b – Chương 4)*

In [ ]:
# pip install scikit-learn pandas
import os
import pandas as pd
from functools import reduce
from sklearn.feature_extraction.text import CountVectorizer

## Đọc dữ liệu

In [ ]:
path_in = "../Data_Preprocessing"   # kết quả tiền xử lý của Câu 1

List_Files = sorted([f for f in os.listdir(path_in) if f.endswith(".txt")])
D = []          # tập văn bản (corpus), mỗi file là 1 văn bản
Doc_Names = []  # tên file tương ứng
for file_name in List_Files:
    f = open(os.path.join(path_in, file_name), "r", encoding="utf-8")
    text = " ".join(f.read().split())   # nối các câu (mỗi dòng 1 câu) thành 1 văn bản
    f.close()
    if text != "":                      # bỏ file rỗng (toàn bộ là stop words)
        D.append(text)
        Doc_Names.append(file_name)

print("Số văn bản:", len(D))
print("Văn bản đầu tiên (" + Doc_Names[0] + "):", D[0])

## Cách 1: Python thuần (Ví dụ 4.2a)

In [ ]:
# B1. Tách từ (dữ liệu đã tách từ ở Câu 1, các từ cách nhau bởi khoảng trắng)
texts = []
for x in D:
    texts.append(x.split())

# B2. Xây dựng từ điển
dictionary = sorted(list(set(reduce(lambda x, y: x + y, texts))))
print("Số từ trong từ điển:", len(dictionary))
print("20 từ đầu tiên:", dictionary[:20])

In [ ]:
# B3. Xây dựng vector BOW
def bag_of_word(sentence):
    vector = []
    # Đếm số lần mỗi từ trong từ điển xuất hiện trong văn bản
    for word in dictionary:
        count = 0
        for w in sentence:
            if w == word:
                count += 1
        vector.append(count)
    return vector

BOW_1 = []
for sentence in texts:
    BOW_1.append(bag_of_word(sentence))

df_bow_1 = pd.DataFrame(BOW_1, columns=dictionary, index=Doc_Names)
print("Kích thước ma trận BOW:", df_bow_1.shape)
print("Vector BOW của văn bản đầu tiên:", BOW_1[0])
df_bow_1.head()

## Cách 2: Sử dụng thư viện sklearn (Ví dụ 4.2b)
Tham số `token_pattern=r"(?u)\S+"`: tách theo khoảng trắng, giữ nguyên từ ghép có `_` và từ 1 ký tự (mặc định sklearn bỏ từ chỉ có 1 ký tự). Dữ liệu đã tiền xử lý nên không cần `lowercase`.

In [ ]:
vect = CountVectorizer(token_pattern=r"(?u)\S+", lowercase=False)
X = vect.fit_transform(D)

# Xây dựng từ điển
dictionary_2 = list(vect.get_feature_names_out())
print("Số từ trong từ điển:", len(dictionary_2))

df_bow_2 = pd.DataFrame(X.toarray(), columns=dictionary_2, index=Doc_Names)
print("Kích thước ma trận BOW:", df_bow_2.shape)
df_bow_2.head()

## So sánh 2 cách và lưu kết quả

In [ ]:
print("Hai cách cho kết quả giống nhau:", df_bow_1.equals(df_bow_2))

# Các từ xuất hiện nhiều nhất trong toàn bộ bộ dữ liệu (tổng theo cột)
print(df_bow_1.sum().sort_values(ascending=False).head(20))

df_bow_1.to_csv("BOW.csv", encoding="utf-8-sig")
print("Đã lưu BOW.csv")